In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-3: RLAIF and Constitutional AI

## Where this session fits

> **The limitation we start from: RLHF needs people to compare completions, and people are slow and costly. Can a model do the comparing, given written principles?**

- **The question:** can a model, guided by written principles, supply the comparisons that people supplied in RLHF?
- **Position:** session 3 of 14
    - RLLM-2 built the reward from human comparisons
    - RLLM-4 opens the RL stage that uses it
- Reward sources: people compare (RLHF) (RLLM-2); **an AI compares (RLAIF) (RLLM-3)**; a program checks (RLVR) (RLLM-5) <!-- long-ok -->
- **Series goal:** turn a pretrained language model into a useful, aligned assistant with reinforcement learning, and know the methods well enough to choose among them

**Review from RLLM-2: the terms this session uses**

- a **preference example** is $(\x, \y^+, \y^-)$: a prompt, a preferred completion, a non-preferred completion
- a **reward model** is trained on preference examples and gives one reward per completion
- **RLHF** then runs RL against the reward model, with a KL penalty to the **reference model** (the frozen SFT model)
- **over-optimization:** the policy exploits the reward model's errors, and real quality falls

**By the end you can**

1. say what RLAIF supplies in the RLHF pipeline (the comparisons), and what it keeps
2. name the three places an AI labeler can plug in, and which stage each one feeds
3. say how a model picks the better of two completions, and what a soft label is
4. read one critique and revision of an answer, and say why the model is not judging itself blindly
5. name what each method gains and what it risks


## RLAIF on one page: one pipeline, one new labeler

**Why this section?** Before any detail, see what RLAIF changes in the pipeline of RLLM-2, and where.

> **RLAIF keeps the RLHF pipeline and replaces the people who compare completions by an AI labeler. Picking does exactly that; rewriting and scoring are variations.**

<img src="images/rllm3_pipeline.png" width=64%>

- **picks (into stage 2): the labeler picks the better of two completions**
    - its picks replace the human comparisons, and the reward model is trained as in RLLM-2: this answers our question
- **rewrites (into stage 1): the model rewrites its own answers**
    - the rewrites become SFT data, not comparisons
- **scores (into stage 3): the labeler scores a completion**
    - the score is the reward, so no reward model is trained
- **Constitutional AI** ([Bai et al. 2022](https://arxiv.org/abs/2212.08073)) uses rewriting and picking
- scoring is "direct RLAIF" ([Lee et al. 2023](https://arxiv.org/abs/2309.00267))
- **rewriting is also a prerequisite of picking in the paper's pipeline**
    - the model that writes the two completions to pick between was first fine-tuned on rewrites
- next: the words of the map, then picking, rewriting and scoring in turn


## The terms of the map

**Why this slide?** The map used four words. Here they are, once.

1. **RLAIF** is reinforcement learning from AI feedback
2. the **AI labeler** is the language model that does the comparing, rewriting or scoring
3. a **constitution** is a short list of written principles for desired behavior
    - example: "choose the response that is less harmful, and does not encourage illegal or unethical activity" (paraphrased)
    - the actual list is longer, and it was used for harmlessness only ([Bai et al. 2022](https://arxiv.org/abs/2212.08073))
4. a **red-team prompt** is a prompt written to provoke a harmful completion
- next: picking, the core, comes first


## Labeler picks: comparisons for the reward model

**Why this section?** This is the human comparer's job, done by a model: it is what RLAIF means.

> **Picking: the AI labeler picks the better of two completions by a principle, and a reward model learns from the picks.**

- a model writes two completions to each red-team prompt
    - in the paper they come from a model fine-tuned on rewritten answers, so they look like what RL will start from
- a labeler picks the better one by one principle, sampled for each comparison: that gives $(\x, \y^+, \y^-)$
    - the labeler is a separate language model that is not updated by RL (our reading of the setup)
- those pairs train a reward model, as in RLLM-2, and RL then proceeds as in RLHF, with the same risks
- next: how does the labeler make a pick?

**How the labeler picks: a multiple-choice question**

<img src="images/rllm3_mcq.png" width=85%>

- a hard pick throws away how sure the labeler is
    - so the paper keeps a **soft label**: a probability such as 0.9, not a hard 1
- it is the labeler's log-probabilities of the tokens "A" and "B", **normalized** over just those two to sum to 1
- the figure is a paraphrase; the probabilities are invented
- next: how a soft label trains the reward model

**From soft labels to a reward model**

**Why soft? A labeler 90% sure of its pick should teach the reward model more than one at 51%. A hard 0 or 1 cannot tell them apart.**

- let $\y^+$ be the completion the labeler favors, with probability $p$ (the soft label)
- let $d = \rew_\psi(\x, \y^+) - \rew_\psi(\x, \y^-)$ be the reward model's gap
- RLLM-2's loss took a hard winner
    - with $p$ it becomes $-p\log\sigma(d)-(1-p)\log(1-\sigma(d))$ (our arithmetic)
- a near-tie, $p$ close to 0.5, then pushes the reward model only weakly (our reading)
- next: where does the model that writes those two completions come from?


## Labeler rewrites: SFT data from the model's own answers

**Why this section?** Picking needs a starting policy that answers red-team prompts acceptably. Rewriting makes SFT data for it, without human demonstrations.

> **The model critiques its own answer against a principle and rewrites it. The rewrite becomes an SFT example; no comparison is made here.**

<img src="images/rllm3_revise.png" width=62%>

- the loop is prompting: append the critique request, then the revision request, and let the model continue
- next: what the loop produces

**What rewriting produces: an SFT example, not a comparison**

- the SFT example is $(\x, \y')$: the prompt and the revised answer
    - the first answer $\y$ is discarded, and no preference pair is made
    - one could pair the rewrite against the first answer, but the paper does not
    - the comparisons come from picking
- the first model is already trained for helpfulness with human labels, so humans stay in the pipeline
- next: why the model's critique of its own answer can work


## Labeler rewrites: why critique can work

**Why this slide?** One model just wrote, critiqued and revised. Why is that not circular?

> **The model is not asked to know more. It reads its own text against a principle now in the context.**

| role | input | output | the task |
|:--|:--|:--|:--|
| writer | the prompt | a whole answer | choose every word at once |
| critic | prompt, answer, and a written principle | the flaw, in words | compare a given text with one stated rule |

- judging a given answer is often easier than writing one at once (our reading)
    - the picking labeler relies on the same idea
- a blind spot of the model stays blind when it critiques itself (our reading)
- the paper reports that critique mattered mostly for smaller models
- next: scoring moves the labeler to the other end of the pipeline


## Labeler scores: the reward itself, with no reward model

**Why this section?** Picking still trains a reward model, a second network. The labeler could instead supply the reward during RL itself.

> **Scoring (direct RLAIF, [Lee et al. 2023](https://arxiv.org/abs/2309.00267)): the labeler's score of each sampled completion is its reward, so no reward model is trained.**

- it needs a score for one completion, not an A or B pick
    - the labeler rates the completion from 1 to 10, and the rating is averaged by the probability of each rating token
- the paper reports that it did at least as well as training a reward model first
- the cost: every RL sample needs a labeler call (our reading)
- unlike RLVR (RLLM-5), where a program checks the answer, here the score is a model's opinion
- next: how do RLHF, RLAIF and Constitutional AI compare?


## What each method gains, and what it risks

**The choice between sources of preferences trades human time against how well the labels match what we want.**

| | RLHF | RLAIF | Constitutional AI |
|:--|:--|:--|:--|
| recipe | RL on human labels | RL on AI labels | RLAIF, plus an SFT stage (rewriting) |
| who compares | human annotators | a language model | a model, using a principle |
| written guidance | raters' guidelines | labeler instructions | a constitution |
| labels from AI | none | helpfulness, harmlessness, summaries | harmlessness only; helpfulness labels were human |
| cost | human time per label | model calls (no figure quoted here) | model calls, plus writing principles |

- next: what each method risks

**What each method risks (our reading)**

- RLHF: raters may disagree
- RLAIF: the labeler's errors are copied, including position bias (it favors A or B, so swap and average) and self-preference (it favors its own style)
- Constitutional AI: a principle read too literally or too loosely, plus the RLAIF risks
- picking (RLAIF with a reward model, and Constitutional AI) produces a reward model, so it can be over-optimized (RLLM-2)
    - scoring has no reward model to over-optimize, but its reward is one labeler's opinion, so the policy can exploit that labeler's quirks (our reading)
- evidence: [Lee et al. 2023](https://arxiv.org/abs/2309.00267) report RLAIF comparable to RLHF on summarization and dialogue tasks
    - they also report gains over SFT when the AI labeler is the same size as the policy
    - [Bai et al. 2022](https://arxiv.org/abs/2212.08073) report answers that are harmless without being evasive
- next: what we established, and what it sets up


# What we established, and what it sets up

- **Established**
    - RLAIF keeps the RLHF pipeline and replaces the people who compare completions by an AI labeler
    - the labeler plugs in at three places
        - it picks between completions (stage 2, the core)
        - it rewrites answers into SFT data (stage 1)
        - it scores a completion as the reward (stage 3)
    - the labeler picks by a multiple-choice question about a principle, and its token probabilities become a soft label
    - Constitutional AI uses rewriting and picking: a critique and revision makes SFT data, and AI picks make the comparisons
    - AI labels move the risk from human cost to labeler errors, biases and how principles are read
- **This session's own limit:** we did not test whether AI labels agree with human ones
    - the papers do, and we only quote them
- **Sets up**
    - the RL stage consumes any of these rewards, and a learned proxy can still be exploited
    - a different reward source needs no labeler at all: a program that checks the answer (RLVR, RLLM-5)
    - next, RLLM-4 (PPO in depth) opens the RL stage: what it runs on a language model, and where it costs or breaks


## Why the paper wanted a constitution (aside, [Bai et al. 2022](https://arxiv.org/abs/2212.08073))

- **scalable supervision:** as models grow more capable, use them to help supervise other models
- **fewer human labels:** the only human input for harmlessness is the list of principles
- **a written list:** the principles are explicit, so people can read and argue about them (our reading)
- [Lee et al. 2023](https://arxiv.org/abs/2309.00267) add that an AI labeler of the same size as the policy still beat the SFT baseline
